# Day 2 — Parse gcd `report_checks` → WNS/TNS

Input: `sta-experiments/gcd/reports/report_checks.rpt` (from `./run_sta.sh`)

**WNS line to explain:** `wns max -1.49` in `opensta_full.rpt` — worst setup slack across all endpoints.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Find repo root (works from notebook dir or repo root)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "sta-experiments" / "parse_timing_report.py").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "sta-experiments"))
from parse_timing_report import parse_report_checks, paths_to_dataframe, compute_wns_tns

REPORT = ROOT / "sta-experiments/gcd/reports/report_checks.rpt"
FULL = ROOT / "sta-experiments/gcd/reports/opensta_full.rpt"
print("ROOT:", ROOT)
print("REPORT:", REPORT)

In [ ]:
text = REPORT.read_text()
paths = parse_report_checks(text)
df = paths_to_dataframe(paths)
df["slack_ns"] = pd.to_numeric(df["slack_ns"], errors="coerce")

wns, tns = compute_wns_tns(df)

# Ground truth from OpenSTA summary (authoritative WNS line)
for line in FULL.read_text().splitlines():
    if line.startswith("wns ") or line.startswith("tns "):
        print(line)

print(f"\nParsed paths: {len(df)}")
print(f"WNS (from paths): {wns:.3f} ns")
print(f"TNS (from paths): {tns:.3f} ns")
df.nsmallest(5, "slack_ns")[["startpoint", "endpoint", "path_group", "slack_ns"]]

In [ ]:
slacks = df["slack_ns"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(slacks, bins=30, edgecolor="black", alpha=0.75)
axes[0].axvline(0, color="red", linestyle="--", label="zero slack")
axes[0].axvline(slacks.min(), color="orange", linestyle=":", label=f"WNS={slacks.min():.3f} ns")
axes[0].set_xlabel("Slack (ns)")
axes[0].set_ylabel("Path count")
axes[0].set_title("gcd setup slack distribution")
axes[0].legend()

wns_by_clk = df.groupby("path_group")["slack_ns"].min().sort_values()
wns_by_clk.plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("WNS (ns)")
axes[1].set_title("WNS by clock group")

fig.suptitle(f"gcd post-route STA — WNS={wns:.3f} ns, TNS={tns:.3f} ns")
fig.tight_layout()
out = ROOT / "sta-experiments/gcd/reports/wns_tns_chart.png"
fig.savefig(out, dpi=120)
plt.show()
print(f"Saved {out}")